# M0 on Colab: smoke → Blank-27 v_c → α sweep

**Purpose:** produce the two artifacts Bridget needs for steered carrier gen:
1. `artifacts/v_c/cat_blank27.pt` (+ `.meta.json`)
2. `results/m0/alpha_sweep_blank27.json` (`selected.alpha`)

**Before you start**
1. In Cursor: install the **Colab** extension (Google) + Jupyter if prompted.
2. Open this notebook → **Select Kernel** → **Colab** → new server → **GPU**.
3. Or open in [colab.research.google.com](https://colab.research.google.com) (Runtime → GPU).
4. **Push** `src/steer/` + scripts to GitHub first, **or** upload a project zip to Drive (clone alone will miss uncommitted code).

Specs: `docs/handoff_M0.md`. Do **not** use smoke artifacts for real M1.

## 0. Check GPU

In [1]:
import torch
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU — Select Kernel → Colab → GPU (or Runtime → Change runtime type)"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

CUDA: True
GPU: Tesla T4
VRAM GiB: 15.6


## 1. Get the repo onto the Colab VM

Colab’s disk is **not** your laptop. Pick **one** of the next two cells.

- **1a** — clone from GitHub (only works if M0 code is already pushed).
- **1b** — mount Drive and use an uploaded zip/folder (works with local uncommitted `src/steer/`).

In [ ]:
# --- 1a: clone from GitHub (skip if using Drive) ---
import os
from pathlib import Path

REPO_URL = "https://github.com/yuen-liu/nlp-final-project.git"  # change if your fork differs
BRANCH = "main"
ROOT = Path("/content/nlp-final-project")

if not ROOT.exists():
    !git clone --branch {BRANCH} {REPO_URL} {ROOT}
else:
    !git -C {ROOT} fetch origin
    !git -C {ROOT} checkout {BRANCH}
    !git -C {ROOT} pull --ff-only origin {BRANCH}

os.chdir(ROOT)
assert (ROOT / "src/steer/generate.py").is_file(), (
    "src/steer missing after clone — push the M0 code to GitHub, or use cell 1b (Drive upload)."
)
print("cwd:", Path.cwd())

In [ ]:
# --- 1b: Drive upload (skip if clone worked) ---
# Upload a zip of the project (including src/steer/) to Drive, then set ZIP_PATH.
from google.colab import drive
from pathlib import Path
import os, zipfile

drive.mount("/content/drive")

ZIP_PATH = Path("/content/drive/MyDrive/nlp-final-project.zip")  # <-- edit
ROOT = Path("/content/nlp-final-project")

if not (ROOT / "src/steer/generate.py").is_file():
    assert ZIP_PATH.is_file(), f"Missing {ZIP_PATH}"
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content")
    # handle zip that contains a top-level folder or flat files
    if not (ROOT / "src/steer/generate.py").is_file():
        candidates = list(Path("/content").glob("**/src/steer/generate.py"))
        assert candidates, "Could not find src/steer after unzip"
        ROOT = candidates[0].parents[2]

os.chdir(ROOT)
print("cwd:", Path.cwd())

## 2. Install deps + download Qwen2.5-7B-Instruct

In [ ]:
%pip install -q -r requirements-gpu.txt

from pathlib import Path
MODEL_DIR = Path("models/Qwen2.5-7B-Instruct")
REV = "a09a35458c702b33eeacc393d103063234e8bc28"

if not any(MODEL_DIR.glob("*.safetensors")):
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    !hf download Qwen/Qwen2.5-7B-Instruct --revision {REV} --local-dir {MODEL_DIR}
else:
    print("model already present:", MODEL_DIR)

print("safetensors:", len(list(MODEL_DIR.glob("*.safetensors"))))

## 3. Smoke (mandatory, cheap)

2-step train + tiny α grid. Smoke files are **not** for Bridget’s lineage.

In [ ]:
!python scripts/gpu_hours.py run --stage vc_train --run-id smoke_blank27 -- \
  python scripts/train_vc.py \
    --prompts data/prompts/steer_train_cat_final.json \
    --out artifacts/v_c/cat_blank27_smoke.pt \
    --steps 2

!python scripts/gpu_hours.py run --stage alpha_sweep --run-id smoke_blank27 -- \
  python scripts/alpha_sweep.py \
    --vector artifacts/v_c/cat_blank27_smoke.pt \
    --out results/m0/alpha_sweep_smoke.json \
    --n-prompts 16 \
    --grid 0 1 2 \
    --batch-size 4

import json
from pathlib import Path
smoke = json.loads(Path("results/m0/alpha_sweep_smoke.json").read_text())
print("smoke selected:", smoke["selected"])
print("SMOKE OK")

## 4. Full Blank-27 v_c (Bridget needs this)

100 steps. Takes longer than smoke; keep the Colab tab awake.

In [ ]:
!python scripts/gpu_hours.py run --stage vc_train --run-id blank27 -- \
  python scripts/train_vc.py \
    --prompts data/prompts/steer_train_cat_final.json \
    --out artifacts/v_c/cat_blank27.pt

from pathlib import Path
import json
meta = json.loads(Path("artifacts/v_c/cat_blank27.meta.json").read_text())
print("norm:", meta.get("norm"), "final_loss:", meta.get("final_loss"))
print("target_token_ids:", meta.get("target_token_ids"))

## 5. Full α sweep for Blank-27 (Bridget needs this)

500 prompts × full α grid. If the session is about to die, re-run **only this cell** after reconnecting (vector file must still exist or re-upload it).

In [ ]:
!python scripts/gpu_hours.py run --stage alpha_sweep --run-id blank27 -- \
  python scripts/alpha_sweep.py \
    --vector artifacts/v_c/cat_blank27.pt \
    --out results/m0/alpha_sweep_blank27.json \
    --batch-size 4

import json
from pathlib import Path
sw = json.loads(Path("results/m0/alpha_sweep_blank27.json").read_text())
print("selected:", sw["selected"])
print("pass rates:", [(r["alpha"], round(r["pass_rate"], 3)) for r in sw["rows"]])

## 6. (Optional) Morgulis-50 reference vector + α

Not required for Bridget’s lineage. Skip if low on time/GPU quota.

In [ ]:
RUN_MORGULIS = False  # set True to run

if RUN_MORGULIS:
    !python scripts/gpu_hours.py run --stage vc_train --run-id morgulis50 -- \
      python scripts/train_vc.py \
        --prompts data/prompts/steer_train_cat_morgulis.json \
        --out artifacts/v_c/cat_morgulis50.pt
    !python scripts/gpu_hours.py run --stage alpha_sweep --run-id morgulis50 -- \
      python scripts/alpha_sweep.py \
        --vector artifacts/v_c/cat_morgulis50.pt \
        --out results/m0/alpha_sweep_morgulis50.json \
        --batch-size 4
else:
    print("skipped Morgulis-50")

## 7. Copy artifacts to Google Drive (hand to Bridget)

Send her at least:
- `cat_blank27.pt`
- `cat_blank27.meta.json`
- `alpha_sweep_blank27.json`

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/nlp_m0_artifacts")
OUT.mkdir(parents=True, exist_ok=True)

files = [
    Path("artifacts/v_c/cat_blank27.pt"),
    Path("artifacts/v_c/cat_blank27.meta.json"),
    Path("results/m0/alpha_sweep_blank27.json"),
]
for f in files:
    assert f.is_file(), f"missing {f} — re-run the cell that creates it"
    dest = OUT / f.name
    shutil.copy2(f, dest)
    print("copied", f, "->", dest)

import json
alpha = json.loads(Path("results/m0/alpha_sweep_blank27.json").read_text())["selected"]["alpha"]
print("\n=== paste to Bridget ===")
print(f"Blank-27 vector + meta + alpha_sweep are in Drive: {OUT}")
print(f"selected.alpha = {alpha}")
!python scripts/gpu_hours.py summary --total

## Done

Bridget can continue from `docs/handoff_Bridget.md` (steered 10k carrier → LoRA M1).

Full `eval_m0` / `readout_m0` are heavy on free Colab — run later on GCP or skip for now.